# meeting_extracted_sem_stage_b — 方向 C Stage B：彙整（PM 審過的條目 → 語意台帳 / 背景情報）

- 用途：`_prompt_weekly_extract.md` v1.1 Stage B 的 Databricks 版，**純規則、無 LLM**。讀鏡頭表 `g_meeting_items_sem_bu` 裡 `verified = 'y'` 且還沒併過（`sem_id IS NULL`）的 semantic / intel 條目，併進 005 的三張台帳表，印出兩份 md（`semantic_record.md` / `background_intel.md` 的表格版）的 diff、變更摘要、未審清單、退場候選。
- 規則（規格 Stage B 1–5，D31）：
  - semantic：`dup_of` 是既有 SEM id → 該 SEM 加一筆證據，不新增；否則發下一個 SEM id（SEM-001 起接著編）＋ 第一筆證據。投影代號用 `route_hint`，PM 在 `review_note` 寫了投影代號（`pa:…` / `skill:…` / `de:…` / `genie:…` / `brief:…` / `mv:…`）就以 review_note 為準。
  - intel：同 app ＋ 同主題（term 正規化後相同）已存在 → 更新 summary / as_of（取較新）/ source / item_key，`n_updates + 1`；否則發 INT id 新增。
  - **D32** `intel_auto = true`（預設）：intel 的 verified 空白也併入，`status = unreviewed`、md 標〔未審〕；PM 之後在鏡頭表標 y → active、該主題全部 n → retired（[c21] 同步）。semantic 仍要 y。
  - **D32** 同文重複（`dup_key` 非空）跟著正本：正本 verified = y 或已併過，重複條目不用標就一起併（semantic 補證據、intel 更新同主題）。
  - event、verified = n、verified 空白（semantic / intel_auto = false 的 intel）：不彙整；空白的列成未審清單。
  - 退場候選：`v_background_intel_q.retire_candidate`（as_of 超過兩季），只列不改。
  - 併過的條目回填鏡頭表 `sem_id`（semantic 填 SEM-xxx、intel 填 INT-xxx），下次不重算。
- 輸入：`g_meeting_items_sem_{unit}`（003）、`g_semantic_record` / `g_semantic_record_evidence` / `g_background_intel`（005）。輸出：同三張表 ＋ 鏡頭表 `sem_id`。
- 參數（widgets）：`catalog`、`schema`、`unit`、`stage_b_version`、`intel_auto`、`dry_run`（預設 true：只印 diff 與摘要，不寫）。
- 排程：每週，PM 審完之後跑；沒新東西就印「無變更」。
- 負責人 / 更新日期：（填）/ 2026-10-19

| cell | 內容 |
|---|---|
| [c01] params | widgets |
| [c02] imports | 常數：id 前綴、投影代號 regex、讀哪些欄 |
| [c03] pure_shared | norm_space / normalize_text / sha1_hex |
| [c04] pure_plan | 發 id、semantic / intel 的併入計畫（純 Python） |
| [c05] pure_render | 兩份 md 的產生與 diff |
| [c07] spark_transforms | 待併列選取、未審統計 |
| [c10] load_source | 讀鏡頭表與台帳（小表，collect） |
| [c11] plan | 算計畫、產 md 前後版 |
| [c12] report | 變更摘要、diff、未審清單、退場候選 |
| [c20] write_ledgers | MERGE 三張表、回填 sem_id |
| [c21] sync_status | unreviewed → active / retired（依 PM 之後的標記） |
| [c30] check_target | 驗證 |


In [ ]:
# [c01] params
# 方向 C Stage B：鏡頭表 verified = y 且 sem_id 空的 semantic / intel → 三張台帳表（005）；印 md diff / 未審 / 退場候選。無 LLM。
# 由 job parameters 覆蓋；改預設值要先 dbutils.widgets.removeAll()。
dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.text("unit", "bu")                 # 鏡頭表 g_meeting_items_sem_{unit}；台帳表不分 unit
dbutils.widgets.text("stage_b_version", "0.2")     # 要等於 [c04] STAGE_B_VERSION_IN_CELL
dbutils.widgets.dropdown("intel_auto", "true", ["true", "false"])  # true：intel 不逐條審，verified 空白也併入、status = unreviewed（D32）
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"])  # true：只印計畫 / diff / 清單，不寫任何表

cfg = {
    "catalog": dbutils.widgets.get("catalog").strip(),
    "schema": dbutils.widgets.get("schema").strip(),
    "unit": dbutils.widgets.get("unit").strip(),
    "stage_b_version": dbutils.widgets.get("stage_b_version").strip(),
    "intel_auto": dbutils.widgets.get("intel_auto") == "true",
    "dry_run": dbutils.widgets.get("dry_run") == "true",
}
assert cfg["catalog"] and cfg["schema"] and cfg["unit"] and cfg["stage_b_version"], "參數不可為空"
print(cfg)

In [ ]:
# [c02] imports
# [c02]～[c05]、[c07] 是純 Python / 純 DataFrame 函式（不碰 spark session / dbutils），tests/ 直接載入。
import difflib
import hashlib
import json
import re
import unicodedata
from collections import Counter, defaultdict
from datetime import date, datetime

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SEM_PREFIX, INT_PREFIX = "SEM-", "INT-"
ID_WIDTH = 3                                   # SEM-001；超過 999 自動變 4 位
ROUTE_DEFAULT = "pa:semantics"                 # 定案條目沒有 route_hint 時的投影代號
ROUTE_RE = re.compile(r"(?<![A-Za-z0-9])(genie|de|skill|pa|brief|mv):[^\s,，;；)）]+")  # review_note 裡的投影代號
AS_OF_RE = re.compile(r"^\d{4}-(?:0[1-9]|1[0-2]|Q[1-4])$")
APP_ORDER = ("TV", "NB", "MNT", "ALL", "OTHER")
RETIRE_QUARTERS = 2                            # 與 005 [c02] 同值；md 標 ⚠ 用
MAX_PENDING = 5000                             # 一次最多併幾條（PM 一週審不到這個數；保護 collect）
MAX_UNREVIEWED_LIST = 30

# 鏡頭表要讀的欄（待併列）
LENS_COLS = ["item_key", "item_id", "week", "category", "app", "customer", "term", "summary", "quote", "date",
             "source", "as_of", "dup_of", "dup_key", "route_hint", "review_note", "verified", "sem_id"]
# select_pending 另加兩欄：pending_reason（verified / intel_auto / dup_of_primary）、eff_verified（本條或正本的 verified）
# 台帳表 Stage B 會更新的欄（其餘：status / pm_note / synced_to / created_at / first_item_key 只在新增時寫，之後是 PM 的）
RECORD_UPDATE_COLS = ["n_evidence", "last_evidence_date", "updated_at"]
INTEL_UPDATE_COLS = ["summary", "as_of", "date", "source", "item_key", "n_updates", "status", "stage_b_version", "updated_at"]
# status 只會從 unreviewed 變 active（PM 標 y）；active / retired 不動（[c04] plan_intel）

In [ ]:
# [c03] pure_shared
# 與 W1 三支 / Stage A [c03] 相同的純函式。複製而不是 import：Databricks 只能貼 cell。
def norm_space(s: str | None) -> str:
    """所有空白（含 U+00A0、換行）壓成一格並去頭尾。"""
    return re.sub(r"\s+", " ", (s or "").replace("\xa0", " ")).strip()


def normalize_text(s: str | None) -> str:
    """去重 / 穩定鍵共用的正規化：NFKC（全半形統一）、去標點、去空白、去控制字元、小寫。"""
    t = unicodedata.normalize("NFKC", s or "")
    return "".join(ch for ch in t if unicodedata.category(ch)[0] not in "PZC").lower()


def sha1_hex(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8")).hexdigest()

In [ ]:
# [c04] pure_plan
# 併入計畫（純 Python；台帳是幾十到幾百列的小表，全在 driver 算）。輸入是鏡頭表待併列與台帳現況，輸出要新增 / 更新的列與回填清單。
STAGE_B_VERSION_IN_CELL = "0.2"  # [c11] 比對 cfg["stage_b_version"]；0.2 = D32（intel_auto、同文重複跟正本）


def parse_id(s: str | None, prefix: str) -> int | None:
    m = re.fullmatch(re.escape(prefix) + r"(\d+)", (s or "").strip())
    return int(m.group(1)) if m else None


def fmt_id(prefix: str, n: int) -> str:
    return f"{prefix}{n:0{ID_WIDTH}d}"


def next_n(ids, prefix: str) -> int:
    """既有 id 的最大流水號 + 1；沒有 → 1。"""
    nums = [parse_id(i, prefix) for i in ids]
    return max([x for x in nums if x is not None] or [0]) + 1


def as_of_key(as_of: str | None) -> int | None:
    """yyyy-Qn / yyyy-MM → 季索引（年×4 + 季−1）；格式不符 None。"""
    s = (as_of or "").strip()
    if not AS_OF_RE.match(s):
        return None
    y = int(s[:4])
    return y * 4 + (int(s[6]) - 1 if s[5] == "Q" else (int(s[5:7]) - 1) // 3)


def as_of_of(item: dict) -> str | None:
    """條目的 as_of：鏡頭表值合格就用；否則由 date 推 yyyy-MM；都沒有 None。"""
    v = (item.get("as_of") or "").strip()
    if AS_OF_RE.match(v):
        return v
    d = item.get("date")
    return f"{d.year:04d}-{d.month:02d}" if isinstance(d, date) else None


def newer_as_of(a: str | None, b: str | None) -> str | None:
    """兩個 as_of 取較新；平手取後者（最新條目）；None 讓位。"""
    ka, kb = as_of_key(a), as_of_key(b)
    if kb is None:
        return a
    if ka is None:
        return b
    return b if kb >= ka else a


def route_from_note(review_note: str | None) -> str | None:
    """PM 在 review_note 寫了投影代號 → 以它為準。"""
    m = ROUTE_RE.search(review_note or "")
    return m.group(0) if m else None


def _max_date(a, b):
    return max([x for x in (a, b) if isinstance(x, date)] or [None])


def _sorted(items: list[dict]) -> list[dict]:
    return sorted(items, key=lambda r: (str(r.get("date") or ""), r["item_key"]))


def _evidence_row(sem_id: str, it: dict, now: datetime) -> dict:
    return {"sem_id": sem_id, "item_key": it["item_key"], "item_id": it.get("item_id"), "quote": it.get("quote"),
            "date": it.get("date"), "source": it.get("source"), "app": it.get("app"), "customer": it.get("customer"),
            "week": it.get("week"), "added_at": now}


def plan_semantic(items: list[dict], records: dict[str, dict], evidence_keys: set[tuple[str, str]],
                  version: str, now: datetime) -> dict:
    """semantic 條目 → 新 SEM / 補證據。records = {sem_id: 台帳列}；evidence_keys = 既有 (sem_id, item_key)。
    去向依序：dup_of（既有或本次新發的 SEM）> primary_sem_id（正本已併）> 本次正本新發的 SEM（dup_key）> 新發 id。
    兩輪：先處理沒有 dup_key 的（正本），再處理同文重複，正本的 id 才能被接到（D32）。"""
    new_records: dict[str, dict] = {}
    updates: dict[str, dict] = {}
    evidence: list[dict] = []
    lens: list[tuple[str, str]] = []
    actions: list[dict] = []
    assigned: dict[str, str] = {}  # item_key → sem_id（本次）
    n = next_n(records, SEM_PREFIX)

    def known(sem_id: str | None) -> bool:
        return bool(sem_id) and (sem_id in records or sem_id in new_records)

    ordered = _sorted([i for i in items if not i.get("dup_key")]) + _sorted([i for i in items if i.get("dup_key")])
    for it in ordered:
        dup_of = (it.get("dup_of") or "").strip() or None
        target = dup_of if known(dup_of) else None
        if dup_of and target is None:
            actions.append({"item_key": it["item_key"], "action": "dup_of_missing", "target": dup_of, "term": it.get("term")})
        if target is None and known(it.get("primary_sem_id")):
            target = it["primary_sem_id"]
        if target is None and it.get("dup_key") in assigned:
            target = assigned[it["dup_key"]]
        if target is None:
            sem_id = fmt_id(SEM_PREFIX, n)
            n += 1
            new_records[sem_id] = {
                "sem_id": sem_id, "term": it.get("term"), "summary": it.get("summary"),
                "route_hint": route_from_note(it.get("review_note")) or it.get("route_hint") or ROUTE_DEFAULT,
                "app": it.get("app") or "ALL", "quote": it.get("quote"), "date": it.get("date"), "source": it.get("source"),
                "first_item_key": it["item_key"], "n_evidence": 1, "last_evidence_date": it.get("date"),
                "status": "confirmed", "synced_to": None, "pm_note": it.get("review_note"),
                "stage_b_version": version, "created_at": now, "updated_at": now,
            }
            evidence.append(_evidence_row(sem_id, it, now))
            assigned[it["item_key"]] = sem_id
            lens.append((it["item_key"], sem_id))
            actions.append({"item_key": it["item_key"], "action": "new_sem", "target": sem_id, "term": it.get("term")})
            continue
        assigned[it["item_key"]] = target
        if (target, it["item_key"]) in evidence_keys:
            lens.append((it["item_key"], target))
            actions.append({"item_key": it["item_key"], "action": "evidence_exists", "target": target, "term": it.get("term")})
            continue
        evidence.append(_evidence_row(target, it, now))
        evidence_keys = evidence_keys | {(target, it["item_key"])}
        rec = new_records.get(target)
        if rec is None:
            base = records[target]
            rec = updates.setdefault(target, {"sem_id": target, "n_evidence": base.get("n_evidence") or 0,
                                              "last_evidence_date": base.get("last_evidence_date"), "updated_at": now})
        rec["n_evidence"] = (rec.get("n_evidence") or 0) + 1
        rec["last_evidence_date"] = _max_date(rec.get("last_evidence_date"), it.get("date"))
        lens.append((it["item_key"], target))
        actions.append({"item_key": it["item_key"], "action": "add_evidence", "target": target, "term": it.get("term")})
    return {"new_records": list(new_records.values()), "updates": list(updates.values()), "evidence": evidence,
            "lens": lens, "actions": actions}


def plan_intel(items: list[dict], intel_rows: list[dict], version: str, now: datetime) -> dict:
    """intel 條目 → 新 INT / 更新同主題。同主題 = 同 app ＋ term 正規化相同（同文重複的 term 本來就相同）。
    status：新列 = active（eff_verified = y）或 unreviewed（intel_auto 併入、PM 還沒標）；
    更新時只會 unreviewed → active（本條 y），active / retired 不動（PM 的）。pm_note 只在新增時寫。"""
    by_norm: dict[tuple[str, str], dict] = {(r.get("app") or "ALL", r.get("term_norm") or normalize_text(r.get("term"))): r
                                            for r in intel_rows}
    new: dict[str, dict] = {}
    touched: dict[str, dict] = {}
    lens: list[tuple[str, str]] = []
    actions: list[dict] = []
    n = next_n([r["intel_id"] for r in intel_rows], INT_PREFIX)
    for it in _sorted(items):
        app = it.get("app") or "ALL"
        key = (app, normalize_text(it.get("term")))
        as_of = as_of_of(it)
        is_y = (it.get("eff_verified") or it.get("verified")) == "y"
        row = by_norm.get(key)
        if row is None:
            intel_id = fmt_id(INT_PREFIX, n)
            n += 1
            row = {"intel_id": intel_id, "app": app, "term": it.get("term"), "term_norm": key[1], "summary": it.get("summary"),
                   "as_of": as_of, "date": it.get("date"), "source": it.get("source"), "item_key": it["item_key"],
                   "first_item_key": it["item_key"], "n_updates": 1, "status": "active" if is_y else "unreviewed",
                   "pm_note": it.get("review_note"), "stage_b_version": version, "created_at": now, "updated_at": now}
            by_norm[key] = row
            new[intel_id] = row
            actions.append({"item_key": it["item_key"], "action": "new_intel", "target": intel_id, "term": it.get("term")})
        else:
            status = "active" if (row.get("status") == "unreviewed" and is_y) else row.get("status")
            row.update({"summary": it.get("summary"), "as_of": newer_as_of(row.get("as_of"), as_of),
                        "date": _max_date(row.get("date"), it.get("date")), "source": it.get("source"),
                        "item_key": it["item_key"], "n_updates": (row.get("n_updates") or 0) + 1, "status": status,
                        "stage_b_version": version, "updated_at": now})
            if row["intel_id"] not in new:
                touched[row["intel_id"]] = row
            actions.append({"item_key": it["item_key"], "action": "update_intel", "target": row["intel_id"], "term": it.get("term")})
        lens.append((it["item_key"], row["intel_id"]))
    return {"new": list(new.values()), "updates": list(touched.values()), "lens": lens, "actions": actions}

In [ ]:
# [c05] pure_render
# 兩份 md（台帳表格版）與 diff。md 只是給人看 / 貼回 PM_Head 的產物，正本是表。
def _c(v) -> str:
    """md 表格 cell：None → ''、| 跳脫、換行壓平。"""
    return norm_space("" if v is None else str(v)).replace("|", "\\|")


def _d(v) -> str:
    return v.isoformat() if isinstance(v, date) else ("" if v is None else str(v))


def quarters_old(as_of: str | None, today: date) -> int | None:
    k = as_of_key(as_of)
    return None if k is None else (today.year * 4 + (today.month - 1) // 3) - k


def render_semantic_md(records: list[dict], evidence_by_sem: dict[str, list[dict]]) -> str:
    """semantic_record.md：定案區表格 ＋ 每條的證據清單 ＋ 退場區。"""
    recs = sorted(records, key=lambda r: parse_id(r.get("sem_id"), SEM_PREFIX) or 0)
    active = [r for r in recs if (r.get("status") or "confirmed") != "retired"]
    retired = [r for r in recs if (r.get("status") or "confirmed") == "retired"]
    out = ["# semantic_record（台帳表格版，由 meeting_extracted_sem_stage_b 產生；正本是 g_semantic_record）", "",
           "## 定案區", "", "| SEM id | app | 主題 | 定義 / 規則 | 投影 | 證據數 | 最新證據 | 同步狀態 |", "|---|---|---|---|---|---|---|---|"]
    out += [f"| {r['sem_id']} | {_c(r.get('app'))} | {_c(r.get('term'))} | {_c(r.get('summary'))} | {_c(r.get('route_hint'))} | "
            f"{r.get('n_evidence') or 0} | {_d(r.get('last_evidence_date'))} | {_c(r.get('synced_to')) or '未投影'} |" for r in active]
    out += ["", "## 證據", ""]
    for r in active:
        out.append(f"- **{r['sem_id']}** {_c(r.get('term'))}")
        for e in sorted(evidence_by_sem.get(r["sem_id"], []), key=lambda e: (_d(e.get("date")), e.get("item_key") or "")):
            out.append(f"  - {_d(e.get('date'))}｜{_c(e.get('source'))}｜「{_c(e.get('quote'))}」")
    out += ["", "## 退場區", ""]
    out += [f"- {r['sem_id']} {_c(r.get('term'))}（{_c(r.get('pm_note'))}）" for r in retired] or ["（無）"]
    return "\n".join(out) + "\n"


def render_intel_md(rows: list[dict], today: date) -> str:
    """background_intel.md：按 app 分節，| as_of | 主題 | 內容 | 來源（id） |；as_of 超過兩季標 ⚠ 退場候選；status = unreviewed 的主題標〔未審〕。"""
    out = ["# background_intel（由 meeting_extracted_sem_stage_b 產生；正本是 g_background_intel）", ""]
    by_app: dict[str, list[dict]] = defaultdict(list)
    for r in rows:
        if (r.get("status") or "active") != "retired":
            by_app[r.get("app") or "ALL"].append(r)
    apps = [a for a in APP_ORDER if a in by_app] + sorted(a for a in by_app if a not in APP_ORDER)
    for app in apps:
        out += [f"## {app}", "", "| as_of | 主題 | 內容 | 來源（id） |", "|---|---|---|---|"]
        for r in sorted(by_app[app], key=lambda r: (-(as_of_key(r.get("as_of")) or 0), _c(r.get("term")))):
            q = quarters_old(r.get("as_of"), today)
            flag = "⚠ " if q is not None and q > RETIRE_QUARTERS else ""
            unrev = "〔未審〕" if r.get("status") == "unreviewed" else ""
            out.append(f"| {flag}{_c(r.get('as_of'))} | {unrev}{_c(r.get('term'))} | {_c(r.get('summary'))} | "
                       f"{_c(r.get('source'))}（{r['intel_id']}） |")
        out.append("")
    if not apps:
        out.append("（無）")
    return "\n".join(out).rstrip() + "\n"


def md_diff(old: str, new: str, name: str) -> str:
    """unified diff；相同回空字串。"""
    lines = list(difflib.unified_diff(old.splitlines(), new.splitlines(), fromfile=f"{name}（現況）", tofile=f"{name}（本次後）",
                                      lineterm="", n=1))
    return "\n".join(lines)

In [ ]:
# [c07] spark_transforms
# 純 DataFrame 函式：待併列（三種來源，D32）、未審統計。不 cache（serverless）。
def select_pending(df_lens: DataFrame, intel_auto: bool) -> DataFrame:
    """待併 = active、sem_id 空、semantic / intel，且符合其一：
    (a) verified = y（pending_reason = verified）；
    (b) intel_auto 且 intel 且 verified 空白（intel_auto）→ 併入但 status = unreviewed；
    (c) 同文重複（dup_key 非空）、verified 空白，而正本 verified = y 或已併過（dup_of_primary）→ 跟著正本。
    輸出 LENS_COLS ＋ pending_reason ＋ eff_verified（(c) 用正本的 verified）＋ primary_sem_id（正本已併時的 id）。
    一個 item_key 只出現一次（(a) 優先）。"""
    base = df_lens.where(
        F.col("active") & F.col("sem_id").isNull() & F.col("category").isin("semantic", "intel")
    )
    null_s = F.lit(None).cast("string")
    a = base.where(F.col("verified") == F.lit("y")).select(
        *LENS_COLS, F.lit("verified").alias("pending_reason"), F.lit("y").alias("eff_verified"),
        null_s.alias("primary_sem_id"))
    primaries = df_lens.where(F.col("active") & ((F.col("verified") == F.lit("y")) | F.col("sem_id").isNotNull())).select(
        F.col("item_key").alias("dup_key"), F.col("verified").alias("_pv"), F.col("sem_id").alias("primary_sem_id"))
    c = (base.where(F.col("verified").isNull() & F.col("dup_key").isNotNull())
         .join(primaries, "dup_key", "inner")
         .select(*LENS_COLS, F.lit("dup_of_primary").alias("pending_reason"), F.col("_pv").alias("eff_verified"),
                 "primary_sem_id"))
    out = a.unionByName(c)
    if intel_auto:
        b = base.where(F.col("verified").isNull() & (F.col("category") == F.lit("intel"))).select(
            *LENS_COLS, F.lit("intel_auto").alias("pending_reason"), null_s.alias("eff_verified"),
            null_s.alias("primary_sem_id"))
        out = out.unionByName(b)
    prio = F.when(F.col("pending_reason") == "verified", 0).when(F.col("pending_reason") == "dup_of_primary", 1).otherwise(2)
    w = Window.partitionBy("item_key").orderBy(prio)
    return out.withColumn("_rn", F.row_number().over(w)).where(F.col("_rn") == 1).drop("_rn")


def unreviewed(df_lens: DataFrame) -> DataFrame:
    """未審清單：active、semantic / intel、verified 空白（n 不算未審）。intel_auto 併進去的仍列（PM 還是可以看）。"""
    return df_lens.where(
        F.col("active") & F.col("verified").isNull() & F.col("category").isin("semantic", "intel")
    ).select("item_key", "category", "app", "customer", "term", "date", "source", "sem_id")

In [ ]:
# [c10] load_source
# 台帳是小表（幾十到幾百列），全部 collect 到 driver 算計畫；鏡頭表只取待併列（上限 MAX_PENDING）。
def fq(name: str) -> str:
    return f"{cfg['catalog']}.{cfg['schema']}.{name}"


fq_lens = fq(f"g_meeting_items_sem_{cfg['unit']}")
fq_rec, fq_evi, fq_intel = fq("g_semantic_record"), fq("g_semantic_record_evidence"), fq("g_background_intel")
fq_intel_q = fq("v_background_intel_q")
for t in (fq_lens, fq_rec, fq_evi, fq_intel, fq_intel_q):
    assert spark.catalog.tableExists(t), f"{t} 不存在：先跑 migration 003 / 005 與 Stage A"

lens_tbl = spark.table(fq_lens)
items = [r.asDict() for r in select_pending(lens_tbl, cfg["intel_auto"]).orderBy("date", "item_key").limit(MAX_PENDING).collect()]
records = {r["sem_id"]: r.asDict() for r in spark.table(fq_rec).collect()}
evidence_rows = [r.asDict() for r in spark.table(fq_evi).collect()]
evidence_keys = {(e["sem_id"], e["item_key"]) for e in evidence_rows}
intel_rows = [r.asDict() for r in spark.table(fq_intel).collect()]
unreviewed_df = unreviewed(lens_tbl)
unreviewed_counts = {r["category"]: r["count"] for r in unreviewed_df.groupBy("category").count().collect()}
unreviewed_sample = [r.asDict() for r in unreviewed_df.orderBy(F.desc("date")).limit(MAX_UNREVIEWED_LIST).collect()]
retire_rows = [r.asDict() for r in spark.table(fq_intel_q).where(F.col("retire_candidate")).orderBy("app", "as_of").collect()]
print(f"pending {len(items)}（semantic {sum(1 for i in items if i['category'] == 'semantic')} / intel "
      f"{sum(1 for i in items if i['category'] == 'intel')}；reason {dict(Counter(i['pending_reason'] for i in items))}）| ledgers: SEM {len(records)} / evidence {len(evidence_rows)} / "
      f"INT {len(intel_rows)} | unreviewed {unreviewed_counts} | retire candidates {len(retire_rows)}")

In [ ]:
# [c11] plan
# 算併入計畫（純 Python），再組「現況」與「本次後」兩版 md 做 diff。不寫表。
assert STAGE_B_VERSION_IN_CELL == cfg["stage_b_version"], (
    f"[c04] 版本 {STAGE_B_VERSION_IN_CELL} ≠ widget stage_b_version {cfg['stage_b_version']}")
now, today = datetime.now(), date.today()
sem_items = [i for i in items if i["category"] == "semantic"]
int_items = [i for i in items if i["category"] == "intel"]
plan_s = plan_semantic(sem_items, records, evidence_keys, cfg["stage_b_version"], now)
plan_i = plan_intel(int_items, [dict(r) for r in intel_rows], cfg["stage_b_version"], now)
actions = plan_s["actions"] + plan_i["actions"]
lens_updates = plan_s["lens"] + plan_i["lens"]

# 現況 md
evi_by_sem: dict[str, list[dict]] = defaultdict(list)
for e in evidence_rows:
    evi_by_sem[e["sem_id"]].append(e)
md_sem_before = render_semantic_md(list(records.values()), evi_by_sem)
md_int_before = render_intel_md(intel_rows, today)

# 本次後 md：既有列套上更新 ＋ 新列
records_after = {k: dict(v) for k, v in records.items()}
for u in plan_s["updates"]:
    records_after[u["sem_id"]].update(u)
for r in plan_s["new_records"]:
    records_after[r["sem_id"]] = r
evi_after = {k: list(v) for k, v in evi_by_sem.items()}
for e in plan_s["evidence"]:
    evi_after.setdefault(e["sem_id"], []).append(e)
intel_after = {r["intel_id"]: dict(r) for r in intel_rows}
for u in plan_i["updates"]:
    intel_after[u["intel_id"]] = u
for r in plan_i["new"]:
    intel_after[r["intel_id"]] = r
md_sem_after = render_semantic_md(list(records_after.values()), evi_after)
md_int_after = render_intel_md(list(intel_after.values()), today)
diff_sem = md_diff(md_sem_before, md_sem_after, "semantic_record.md")
diff_int = md_diff(md_int_before, md_int_after, "background_intel.md")
print(f"actions {len(actions)}：", dict(Counter(a["action"] for a in actions)))

In [ ]:
# [c12] report
# 規格 Stage B 第 5 點的輸出：(a) 兩份 md 的變更 (b) 變更摘要表 (c) 未審清單 (d) 退場候選。沒有變更就明說。
print("=" * 30, "變更摘要（item_key → 動作 → 去向）", "=" * 30)
if not actions:
    print("本次無變更：沒有 verified = y 且未併的 semantic / intel 條目。")
for a in actions:
    print(f"{a['item_key'][:10]}  {a['action']:<16} {a['target']:<8} {norm_space(a.get('term'))[:40]}")
missing = [a for a in actions if a["action"] == "dup_of_missing"]
if missing:
    print(f"\nWARN {len(missing)} 條的 dup_of 指向不存在的 SEM id，已當新條目處理：", [(a["item_key"][:10], a["target"]) for a in missing])

print("\n" + "=" * 30, "semantic_record.md 變更", "=" * 30)
print(diff_sem or "（無變更）")
print("\n" + "=" * 30, "background_intel.md 變更", "=" * 30)
print(diff_int or "（無變更）")

print("\n" + "=" * 30, f"未審清單（verified 空白；semantic / intel）{unreviewed_counts}", "=" * 30)
for r in unreviewed_sample:
    print(f"{r['item_key'][:10]}  {r['category']:<9} {r['app'] or '-':<4} {r['customer'] or '-':<12} {r['date']}  "
          f"{norm_space(r['term'])[:40]}{'  （已自動併入 ' + r['sem_id'] + '）' if r.get('sem_id') else ''}")
if sum(unreviewed_counts.values()) > len(unreviewed_sample):
    print(f"…共 {sum(unreviewed_counts.values())} 條，只列最新 {len(unreviewed_sample)} 條；全表看 v_meeting_items_sem_q WHERE verified IS NULL")

print("\n" + "=" * 30, f"退場候選（as_of 超過 {RETIRE_QUARTERS} 季，待 PM 確認；不自動改）", "=" * 30)
for r in retire_rows:
    print(f"{r['intel_id']}  {r['app']:<4} {r['as_of']:<8} {r['quarters_old']} 季  {norm_space(r['term'])[:40]}")
if not retire_rows:
    print("（無）")
if cfg["dry_run"]:
    print("\ndry_run = true：以上只是計畫，沒有寫任何表。確認後 dry_run = false 重跑。")

In [ ]:
# [c20] write_ledgers
# 四個寫入，全部以主鍵 MERGE / append：(1) g_semantic_record 新列 append、既有列只更新 RECORD_UPDATE_COLS；
# (2) evidence append（鍵 (sem_id, item_key) 已在 [c04] 排除重複）；(3) g_background_intel 新列 append、既有列更新 INTEL_UPDATE_COLS
# （status / pm_note 不動）；(4) 鏡頭表回填 sem_id（semantic = SEM-xxx、intel = INT-xxx），下次不重算。
def _df_for(fq_table: str, rows: list[dict]):
    schema = spark.table(fq_table).schema
    return spark.createDataFrame([tuple(r.get(f.name) for f in schema.fields) for r in rows], schema)


def _merge_update(fq_table: str, key: str, rows: list[dict], cols: list[str], view: str) -> None:
    _df_for(fq_table, rows).select(key, *cols).createOrReplaceTempView(view)
    spark.sql(f"MERGE INTO {fq_table} t USING {view} s ON t.{key} = s.{key} "
              f"WHEN MATCHED THEN UPDATE SET " + ", ".join(f"t.{c} = s.{c}" for c in cols))


n_written = {}
if not cfg["dry_run"] and lens_updates:
    if plan_s["new_records"]:
        _df_for(fq_rec, plan_s["new_records"]).write.mode("append").saveAsTable(fq_rec)
    if plan_s["updates"]:
        _merge_update(fq_rec, "sem_id", plan_s["updates"], RECORD_UPDATE_COLS, "_sb_rec_upd")
    if plan_s["evidence"]:
        _df_for(fq_evi, plan_s["evidence"]).write.mode("append").saveAsTable(fq_evi)
    if plan_i["new"]:
        _df_for(fq_intel, plan_i["new"]).write.mode("append").saveAsTable(fq_intel)
    if plan_i["updates"]:
        _merge_update(fq_intel, "intel_id", plan_i["updates"], INTEL_UPDATE_COLS, "_sb_intel_upd")
    spark.createDataFrame(lens_updates, "item_key string, sem_id string").createOrReplaceTempView("_sb_lens_upd")
    spark.sql(f"MERGE INTO {fq_lens} t USING _sb_lens_upd s ON t.item_key = s.item_key "
              "WHEN MATCHED THEN UPDATE SET t.sem_id = s.sem_id, t.updated_at = current_timestamp()")
    n_written = {"sem_new": len(plan_s["new_records"]), "sem_updated": len(plan_s["updates"]), "evidence": len(plan_s["evidence"]),
                 "intel_new": len(plan_i["new"]), "intel_updated": len(plan_i["updates"]), "lens_sem_id": len(lens_updates)}
    print("written:", n_written)
else:
    print("skip write（dry_run 或無變更）")

In [ ]:
# [c21] sync_status
# D32：intel_auto 併進去的主題 status = unreviewed。PM 之後在鏡頭表標 verified：該主題任一條目 y → active；
# 全部標 n（沒有 y、沒有空白）→ retired（pm_note 記「PM 全標 n」）。只動 unreviewed 的列；active / retired 是 PM 的，不碰。
# 每次跑都做（不限本批），dry_run 只印不寫。
sync_sql = f"""
    SELECT sem_id AS intel_id,
           max(CASE WHEN verified = 'y' THEN 1 ELSE 0 END) AS has_y,
           min(CASE WHEN verified = 'n' THEN 1 ELSE 0 END) AS all_n
    FROM {fq_lens} WHERE active AND sem_id LIKE 'INT-%' GROUP BY sem_id"""
to_sync = spark.sql(f"""
    SELECT t.intel_id, s.has_y, s.all_n FROM {fq_intel} t JOIN ({sync_sql}) s ON t.intel_id = s.intel_id
    WHERE t.status = 'unreviewed' AND (s.has_y = 1 OR s.all_n = 1)""").collect()
n_to_active = sum(1 for r in to_sync if r["has_y"] == 1)
n_to_retired = sum(1 for r in to_sync if r["has_y"] == 0 and r["all_n"] == 1)
if not cfg["dry_run"] and to_sync:
    spark.sql(f"""
        MERGE INTO {fq_intel} t USING ({sync_sql}) s ON t.intel_id = s.intel_id AND t.status = 'unreviewed'
        WHEN MATCHED AND s.has_y = 1 THEN UPDATE SET t.status = 'active', t.updated_at = current_timestamp()
        WHEN MATCHED AND s.all_n = 1 THEN UPDATE SET t.status = 'retired',
             t.pm_note = coalesce(t.pm_note, 'PM 全標 n（Stage B 自動退場）'), t.updated_at = current_timestamp()
    """)
print(f"status sync（unreviewed → active {n_to_active} / → retired {n_to_retired}）" + ("（dry_run 不寫）" if cfg["dry_run"] else ""))

In [ ]:
# [c30] check_target
# 寫入後驗證：三張台帳主鍵無重複；本次回填的 item_key 都有 sem_id；每個動過的 SEM 的 n_evidence = 證據列數；
# 新 INT 的 term_norm 在同 app 內唯一。
if not cfg["dry_run"] and lens_updates:
    rec, evi, intel, lens = spark.table(fq_rec), spark.table(fq_evi), spark.table(fq_intel), spark.table(fq_lens)
    touched = [r["sem_id"] for r in plan_s["new_records"]] + [u["sem_id"] for u in plan_s["updates"]]
    keys_df = spark.createDataFrame([(k,) for k, _ in lens_updates], "item_key string")
    evi_cnt = {r["sem_id"]: r["n"] for r in evi.where(F.col("sem_id").isin(*touched)).groupBy("sem_id").agg(F.count("*").alias("n")).collect()} if touched else {}
    rec_cnt = {r["sem_id"]: r["n_evidence"] for r in rec.where(F.col("sem_id").isin(*touched)).select("sem_id", "n_evidence").collect()} if touched else {}
    chk = {
        "dup_sem_id": rec.groupBy("sem_id").count().where("count > 1").count(),
        "dup_intel_id": intel.groupBy("intel_id").count().where("count > 1").count(),
        "dup_evidence_key": evi.groupBy("sem_id", "item_key").count().where("count > 1").count(),
        "dup_intel_topic": intel.groupBy("app", "term_norm").count().where("count > 1").count(),
        "lens_filled": lens.join(keys_df, "item_key", "left_semi").where(F.col("sem_id").isNotNull()).count(),
        "lens_expected": len(lens_updates),
        "evidence_mismatch": [s for s in touched if evi_cnt.get(s, 0) != (rec_cnt.get(s) or 0)],
        "unreviewed_with_y": intel.where(F.col("status") == "unreviewed").join(
            lens.where((F.col("verified") == "y") & F.col("active")).select(F.col("sem_id").alias("intel_id")).distinct(),
            "intel_id", "left_semi").count(),
        "totals": {"sem": rec.count(), "evidence": evi.count(), "intel": intel.count()},
    }
    print(chk)
    assert chk["dup_sem_id"] == 0 and chk["dup_intel_id"] == 0 and chk["dup_evidence_key"] == 0, "台帳主鍵重複"
    assert chk["dup_intel_topic"] == 0, "同 app 同主題有兩列（term_norm 比對壞了）"
    assert chk["lens_filled"] == chk["lens_expected"], "鏡頭表 sem_id 回填不完整"
    assert not chk["evidence_mismatch"], f"n_evidence 與證據列數不符：{chk['evidence_mismatch']}"
    assert chk["unreviewed_with_y"] == 0, "有 PM 標 y 的 intel 仍是 unreviewed（[c21] 同步沒做到）"